In [13]:
# !pip install xgboost onnxruntime pandas
!pip install joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 307.7/307.7 kB 5.7 MB/s eta 0:00:00a 0:00:01


In [8]:
ls /mnt/object/LoanData/val

template_test/  val_templatetest.csv  val_transformed.csv


In [9]:
import time
import numpy as np
import pandas as pd
import xgboost as xgb
import onnxruntime as ort
import os

# Paths
pth_model_path = "/home/jovyan/work/train/models/xgb-kfold-binary-13/model.pth"
# onnx_model_path = "/mnt/block/models/model.onnx"
eval_csv_path = "/mnt/object/LoanData/val/val_transformed.csv"


In [10]:
df = pd.read_csv(eval_csv_path)
if 'risk_level' in df.columns:
    y = df['risk_level']
    X = df.drop(columns=['risk_level'])
else:
    X = df
X_np = X.astype(np.float32).to_numpy()
print("Input shape:", X_np.shape)


Input shape: (203460, 68)


In [14]:
import joblib
import numpy as np
import time

# Load model
model = joblib.load(pth_model_path)  # or your .pth path if you used joblib

# Make sure X_np is a NumPy array of shape (n_samples, n_features)
# X_np = ...

# Single sample latency
latencies = []
for _ in range(100):
    start = time.time()
    _ = model.predict(X_np[0:1])
    latencies.append(time.time() - start)
print(f"XGBoost Latency (median): {np.median(latencies)*1000:.2f} ms")
print(f"XGBoost Latency (95th pct): {np.percentile(latencies,95)*1000:.2f} ms")

# Batch throughput
batch_start = time.time()
_ = model.predict(X_np)
batch_duration = time.time() - batch_start
fps = len(X_np) / batch_duration
print(f"XGBoost Throughput: {fps:.2f} samples/sec")


XGBoost Latency (median): 0.79 ms
XGBoost Latency (95th pct): 0.87 ms
XGBoost Throughput: 330803.61 samples/sec


In [ ]:
session = ort.InferenceSession(onnx_model_path, providers=["OpenVINOExecutionProvider"])
input_name = session.get_inputs()[0].name

# Single sample latency
latencies = []
for _ in range(100):
    start = time.time()
    _ = session.run(None, {input_name: X_np[0:1]})
    latencies.append(time.time() - start)
print(f"ONNX Latency (median): {np.median(latencies)*1000:.2f} ms")
print(f"ONNX Latency (95th pct): {np.percentile(latencies,95)*1000:.2f} ms")

# Batch throughput
start = time.time()
_ = session.run(None, {input_name: X_np})
duration = time.time() - start
fps = len(X_np) / duration
print(f"ONNX Throughput: {fps:.2f} samples/sec")
